In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 05_facts/05_fact_procedures
# Purpose  : fact_procedures - Grain: One row = one medical procedure.

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.fact_procedures",
    comment="Fact table - one row per medical procedure performed",
)
def fact_procedures():
    procedures = dlt.read_stream("covid19_lakehouse.silver.procedures")

    patients = (
        spark.read.table("covid19_lakehouse.gold.dim_patient")
        .filter(F.col("is_current") == True)  # noqa: E712
        .select(F.col("patient_id"), F.col("patient_key"))
    )
    procedure_dim = spark.read.table("covid19_lakehouse.gold.dim_procedure").select(
        F.col("procedure_code"), F.col("procedure_key")
    )
    reasons = spark.read.table("covid19_lakehouse.gold.dim_reason").select(
        F.col("reason_code"), F.col("reason_key")
    )

    df = (
        procedures
        .join(patients, on="patient_id", how="left")
        .join(procedure_dim, on="procedure_code", how="left")
        .join(reasons, on="reason_code", how="left")
    )

    return df.select(
        F.xxhash64("patient_id", "encounter_id", "procedure_code", "procedure_date").alias(
            "procedure_fact_key"
        ),
        F.col("patient_key"),
        F.col("encounter_id"),  # Degenerate Dimension
        F.col("procedure_key"),
        F.coalesce(F.col("reason_key"), F.lit(-1)).alias("reason_key"),
        F.date_format(F.col("procedure_date"), "yyyyMMdd").cast("int").alias("date_key"),
        F.col("base_cost"),
    )